In [232]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.v2
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

import os
import pathlib
import PIL.Image
import pandas as pd
import seaborn as sns
import sklearn.model_selection
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
!echo Dataset from: (cat /opt/datasets/imagenet-256-dimensi0n/kaggle.url)

Dataset from: https://www.kaggle.com/datasets/dimensi0n/imagenet-256


In [3]:
DATASET_ROOT = pathlib.Path("/opt/datasets/imagenet-256-dimensi0n/")

[directory.stem for directory in DATASET_ROOT.glob(pattern="*") if directory.is_dir()][:10]

['quilt',
 'snowplow',
 'gong',
 'teddy',
 'birdhouse',
 'projectile',
 'convertible',
 'trombone',
 'apiary',
 'rock_beauty']

In [4]:
def make_train_test_split(root, **kwargs):
    assert isinstance(root, pathlib.Path)

    paths, labels = (
        [str(path) for path in root.rglob("*/*.jpg")],
        [path.parent.stem for path in root.rglob("*/*.jpg")]
    )

    samples = pd.DataFrame(zip(paths, labels), columns=["path", "label"])

    samples_train, samples_test = sklearn.model_selection.train_test_split(
        samples,
        **kwargs,
        stratify=labels
    )

    return samples_train, samples_test

In [5]:
dataframe_train, dataframe_test = make_train_test_split(DATASET_ROOT, train_size=0.9)
len(dataframe_train), len(dataframe_test)

(485843, 53983)

In [216]:
class ImageNetDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, transform=None, target_transform=None):
        super().__init__()

        self.dataframe = dataframe
        self.transform = transform or (lambda sample: sample)
        self.target_transform = target_transform or (lambda sample: sample)

        self.class_to_idx = {
            category: index
            for (index, category) in enumerate(
                sorted(self.dataframe["label"].unique())
            )
        }

    def __getitem__(self, index):
        path, label = self.dataframe.iloc[index]
        image, label = (
            self.transform(PIL.Image.open(path).convert(mode="RGB")),
            self.target_transform(self.class_to_idx[label])
        )
        return image, label

    def __len__(self):
        return len(self.dataframe)
    
dataset_train = ImageNetDataset(dataframe=dataframe_train)
dataset_train[0]

(<PIL.Image.Image image mode=RGB size=256x256>, 543)

In [217]:
transform_std, transform_mean = torch.tensor([[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]])

transform_extra = transforms.Compose([
    transforms.RandomResizedCrop(size=256, scale=[0.08, 1], ratio=[3/4, 4/3]),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandAugment(num_ops=2, magnitude=8),
    transforms.ToTensor(),
    transforms.Normalize(mean=transform_mean, std=transform_std, inplace=True),
    transforms.RandomErasing(p=0.25, scale=[0.02, 0.2], ratio=[0.3, 3.3], inplace=True)
])

transform_basic = transforms.Compose([
    transforms.Resize(size=256),
    transforms.CenterCrop(size=[256, 256]),
    transforms.ToTensor(),
    transforms.Normalize(mean=transform_mean, std=transform_std, inplace=True)
])

dataset_train = ImageNetDataset(dataframe=dataframe_train, transform=transform_extra)
dataset_test = ImageNetDataset(dataframe=dataframe_test, transform=transform_basic)

dataloader_train = torch.utils.data.DataLoader(
    dataset=dataset_train,
    batch_size=64,
    shuffle=True,
    num_workers=8
)

dataloader_test = torch.utils.data.DataLoader(
    dataset=dataset_test,
    batch_size=128,
    shuffle=False,
    num_workers=8
)

In [302]:
X_batch, y_batch = next(iter(dataloader_train))
X_batch.shape, y_batch.shape

(torch.Size([64, 3, 256, 256]), torch.Size([64]))

In [303]:
transform_mixup = torchvision.transforms.v2.MixUp(
    alpha=0.2,
    num_classes=len(dataset_train.class_to_idx)
)
X_batch, y_batch = transform_mixup(X_batch, y_batch)
X_batch.shape, y_batch.shape

(torch.Size([64, 3, 256, 256]), torch.Size([64, 1000]))

In [304]:
transform_cutmix = torchvision.transforms.v2.CutMix(
    alpha=1.0,
    num_classes=len(dataset_train.class_to_idx)
)
X_batch, y_batch = transform_cutmix(X_batch, y_batch)
X_batch.shape, y_batch.shape

(torch.Size([64, 3, 256, 256]), torch.Size([64, 1000]))